In [ ]:
!pip install opencv-python tqdm pillow

In [ ]:
!aws s3 cp s3://object-detection-data-s3/raw/leftImg8bit_trainvaltest.zip .

In [ ]:
!aws s3 cp s3://object-detection-data-s3/raw/gtFine_trainvaltest.zip .

In [ ]:
import zipfile
import os

def extract(zip_path, out_dir):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(out_dir)

os.makedirs("cityscapes", exist_ok=True)

extract("leftImg8bit_trainvaltest.zip", "cityscapes")
extract("gtFine_trainvaltest.zip", "cityscapes")

In [ ]:
CLASS_MAP = {
    "pole": 0,
    "fence": 1,
    "vegetation": 2,
    "terrain": 3
}

In [ ]:
import json
import cv2
import shutil
from tqdm import tqdm
import os

img_dir = "cityscapes/leftImg8bit"
ann_dir = "cityscapes/gtFine"

# Create output dirs for both train and val splits
for split in ["train", "val"]:
    os.makedirs(f"dataset/images/{split}", exist_ok=True)
    os.makedirs(f"dataset/labels/{split}", exist_ok=True)


def polygon_to_yolo_seg(poly, img_w, img_h):
    seg = []
    for x, y in poly:
        seg.append(x / img_w)
        seg.append(y / img_h)
    return seg


def process_split(split):
    split_ann_dir = os.path.join(ann_dir, split)
    split_img_dir = os.path.join(img_dir, split)

    if not os.path.exists(split_ann_dir):
        print(f"Split '{split}' not found, skipping.")
        return

    for city in os.listdir(split_ann_dir):
        city_path = os.path.join(split_ann_dir, city)

        for file in tqdm(os.listdir(city_path), desc=f"{split}/{city}"):
            if not file.endswith("gtFine_polygons.json"):
                continue

            json_path = os.path.join(city_path, file)
            with open(json_path) as f:
                data = json.load(f)

            # Early class filter — skip image load if no relevant objects
            relevant = [obj for obj in data["objects"] if obj["label"] in CLASS_MAP]
            if not relevant:
                continue

            img_name = file.replace("gtFine_polygons.json", "leftImg8bit.png")
            img_path = os.path.join(split_img_dir, city, img_name)

            if not os.path.exists(img_path):
                continue

            # Read image only to get dimensions
            img = cv2.imread(img_path)
            if img is None:  # handle corrupt/missing images
                print(f"Warning: could not read {img_path}, skipping.")
                continue
            h, w, _ = img.shape

            yolo_lines = []
            for obj in relevant:
                poly = obj["polygon"]
                if len(poly) < 3:
                    continue
                class_id = CLASS_MAP[obj["label"]]
                seg = polygon_to_yolo_seg(poly, w, h)
                yolo_lines.append(str(class_id) + " " + " ".join(map(str, seg)))

            if not yolo_lines:
                continue

            # Copy image directly (no re-encoding)
            shutil.copy(img_path, f"dataset/images/{split}/{img_name}")

            # Write label file
            lbl_path = f"dataset/labels/{split}/{img_name.replace('.png', '.txt')}"
            with open(lbl_path, "w") as f:
                f.write("\n".join(yolo_lines))


process_split("train")
process_split("val")

In [ ]:
yaml_content = """
path: dataset

train: images/train
val: images/val

names:
  0: pole
  1: fence
  2: vegetation
  3: terrain
"""

with open("data.yaml", "w") as f:
    f.write(yaml_content)

print("data.yaml written successfully.")

In [ ]:
# Save processed dataset back to S3
!aws s3 cp -r dataset/ s3://object-detection-data-s3/yolo-dataset/
!aws s3 cp data.yaml s3://object-detection-data-s3/yolo-dataset/
print("Dataset successfully uploaded to S3.")